# H3: Normalized SGD vs Muon

This notebook is the notebook-facing counterpart to `run_experiment.py` in the same experiment directory.
It uses the **shared script implementation** rather than re-implementing the training core here.

## Pair-level goal of this notebook

Evaluate, in a fixed toy synthetic setting, whether **removing update scale alone** can reproduce the
combination of basin/diversity behavior and optimization behavior seen with Muon-style orthogonalized
updates.

### What is compared

- SGD
- Muon-style Newton-Schulz orthogonalized updates
- Frobenius-normalized SGD
- Spectral-norm-normalized SGD
- Sign SGD

### What is actually measured here

- selected learning rate from a fixed grid search
- full loss curves across 20 seeded runs
- final loss mean and spread across runs
- weight-space diversity and function-space diversity at convergence
- F/W ratio = function diversity / weight diversity
- per-layer conditioning summaries
- a small set of **heuristic comparison rules**

### Scope and caveats

This remains a **controlled toy experiment** on fixed 4-layer 32x32 networks with a fixed synthetic target.
The notebook therefore reports **descriptive statistics and heuristic adjudication rules**, not formal
hypothesis tests and not a universal answer to whether the polar factor is necessary in general.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text


def find_script_path(explicit_path=None):
    candidates = []
    if explicit_path is not None:
        candidates.append(Path(explicit_path).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd / 'run_experiment.py',
        cwd / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'H3_NORMALIZED_SGD_vs_MUON' / 'run_experiment.py',
        cwd.parent / 'run_experiment.py',
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(
        'Could not locate run_experiment.py. Checked:\n' + searched +
        '\nSet explicit_path in find_script_path(...) if running from an unusual working directory.'
    )


SCRIPT_PATH = find_script_path()
NOTEBOOK_OUTPUT_DIR = SCRIPT_PATH.parent / 'notebook_outputs'

spec = importlib.util.spec_from_file_location('h3_normalized_sgd_vs_muon', SCRIPT_PATH)
h3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h3)


def show_table(rows, columns=None, caption=None):
    if caption:
        print(caption)
    if pd is not None:
        df = pd.DataFrame(rows)
        if columns is not None:
            df = df[columns]
        display(df)
        return df

    if columns is None and rows:
        columns = list(rows[0].keys())
    if rows:
        print(' | '.join(columns))
        print('-' * 100)
        for row in rows:
            print(' | '.join(str(row[col]) for col in columns))
    else:
        print('(no rows)')
    return rows


print('Counterpart script:', SCRIPT_PATH)
print('Notebook output directory:', NOTEBOOK_OUTPUT_DIR)
print('Notebook policy: use shared script code for execution and result structures; do not duplicate the experiment core here.')

## Reproducibility and runtime configuration

The next cell logs the exact experiment identity, default configuration, search grids, seeds, and basic
runtime environment. This notebook does **not** rely on `__file__`; instead it resolves the counterpart
script path explicitly and chooses an explicit output directory for saved figures.

In [ ]:
config = h3.get_config(output_dir=NOTEBOOK_OUTPUT_DIR)

try:
    import matplotlib
    matplotlib_version = matplotlib.__version__
except Exception:
    matplotlib_version = 'unavailable'

print('Experiment ID:        ', config['experiment_id'])
print('Title:                ', config['title'])
print('Scope note:           ', config['scope_note'])
print('Script path:          ', config['script_path'])
print('Output directory:     ', config['resolved_output_dir'])
print('Python version:       ', sys.version.split()[0])
print('Platform:             ', platform.platform())
print('NumPy version:        ', h3.np.__version__)
print('matplotlib version:   ', matplotlib_version)
print('Git commit:           ', h3._get_git_commit())
print('Architectures:        ', config['architectures'])
print('Optimizers:           ', config['optimizers'])
print('Data seed:            ', config['data_seed'])
print('Run seeds:            ', config['run_seeds'][0], '...', config['run_seeds'][-1])
print('LR sweep steps:       ', config['lr_sweep_steps'])
print('Training steps:       ', config['num_steps'])
print('Independent runs:     ', config['num_independent_runs'])
print('Expected runtime note:', 'full default run is typically around one minute on CPU in this environment')

lr_grid_rows = [
    {
        'optimizer': h3.OPTIMIZER_LABELS[opt],
        'candidate_grid': config['lr_candidate_grids'][opt],
    }
    for opt in h3.OPTIMIZER_NAMES
]
show_table(lr_grid_rows, columns=['optimizer', 'candidate_grid'], caption='Learning-rate candidate grids')

## Run the shared experiment pipeline

This cell runs the exact script-side experiment function, saves the standard figures to the explicit notebook
output directory, and returns a structured results bundle with raw arrays and heuristic summaries for use below.

In [ ]:
results = h3.run_full_experiment(
    output_dir=NOTEBOOK_OUTPUT_DIR,
    make_plots=True,
    verbose=True,
)

print(f"\nMeasured runtime: {results['metadata']['runtime_seconds']:.1f}s")
print('Saved figure paths:')
for key, value in results['figure_paths'].items():
    print(f'  {key}: {value}')

## Learning-rate selection summary

The search below is still the original fixed-grid 200-step sweep from the script. A grid-edge selection is a
useful warning sign: it means the best observed learning rate sat at the boundary of the tested grid, so a wider
search could plausibly change the selected value.

In [ ]:
lr_rows = h3.build_lr_summary_rows(results)
show_table(
    lr_rows,
    columns=[
        'architecture',
        'optimizer',
        'selected_lr',
        'best_200_step_loss',
        'hit_grid_boundary',
        'boundary_side',
        'num_candidates',
    ],
    caption='Selected learning rates by architecture and optimizer',
)

boundary_rows = [row for row in lr_rows if row['hit_grid_boundary']]
if boundary_rows:
    show_table(
        boundary_rows,
        columns=['architecture', 'optimizer', 'selected_lr', 'boundary_side'],
        caption='Boundary-hit learning-rate selections (interpret cautiously)',
    )
else:
    print('No selected learning rate hit the edge of its search grid.')

## Deep linear network

The linear case is the cleanest place to ask whether simple normalization can reproduce Muon-like basin behavior.
The figure shows mean loss curves with spread across runs, then descriptive diversity summaries and the two current
paradox-style heuristics used by the script.

In [ ]:
h3.plot_architecture_overview(results, 'linear', show=True)

In [ ]:
show_table(
    h3.build_optimizer_summary_rows(results, 'linear'),
    columns=[
        'optimizer',
        'selected_lr',
        'final_loss_mean',
        'final_loss_std',
        'function_over_weight_ratio',
        'condition_geom_mean',
        'condition_geom_std',
        'stable_run_fraction',
        'num_diverged_runs',
    ],
    caption='Linear architecture: final loss, F/W ratio, and conditioning summary',
)

show_table(
    h3.build_paradox_metric_rows(results, 'linear'),
    columns=[
        'optimizer',
        'weight_diversity_mean',
        'func_diversity_mean',
        'function_over_weight_ratio',
        'paradox_strength',
        'final_loss_std',
    ],
    caption='Linear architecture: paradox-related descriptive metrics',
)

show_table(
    h3.build_heuristic_rule_rows(results, 'linear'),
    columns=['rule', 'description', 'passed', 'summary'],
    caption='Linear architecture: heuristic rule outcomes',
)

print('Linear interpretation:')
print(results['heuristic_results']['linear']['critical_comparison']['conclusion'])

**Linear-reading guide.**

If normalized SGD lowers the F/W ratio relative to plain SGD, that supports the idea that magnitude removal can
change the basin signature. But the more stringent question is whether it also reaches Muon-comparable loss.
That distinction matters: reproducing one descriptive signature is not the same as reproducing the optimization
behavior of the orthogonalized update.

## ReLU network

The ReLU case deliberately tests whether the same story survives when the exact deep-linear gauge symmetry is broken.
This is where the notebook should be especially careful not to over-generalize from the linear case.

In [ ]:
h3.plot_architecture_overview(results, 'relu', show=True)

In [ ]:
show_table(
    h3.build_optimizer_summary_rows(results, 'relu'),
    columns=[
        'optimizer',
        'selected_lr',
        'final_loss_mean',
        'final_loss_std',
        'function_over_weight_ratio',
        'condition_geom_mean',
        'condition_geom_std',
        'stable_run_fraction',
        'num_diverged_runs',
    ],
    caption='ReLU architecture: final loss, F/W ratio, and conditioning summary',
)

show_table(
    h3.build_paradox_metric_rows(results, 'relu'),
    columns=[
        'optimizer',
        'weight_diversity_mean',
        'func_diversity_mean',
        'function_over_weight_ratio',
        'paradox_strength',
        'final_loss_std',
    ],
    caption='ReLU architecture: paradox-related descriptive metrics',
)

show_table(
    h3.build_heuristic_rule_rows(results, 'relu'),
    columns=['rule', 'description', 'passed', 'summary'],
    caption='ReLU architecture: heuristic rule outcomes',
)

print('ReLU interpretation:')
print(results['heuristic_results']['relu']['critical_comparison']['conclusion'])

**ReLU-reading guide.**

Here the most important question is whether the normalization baselines still track Muon on loss and on the
paradox-related descriptive summaries. If the rankings split, the notebook should say so explicitly rather than
pretend there is a single unambiguous paradox ordering.

## Cross-architecture comparison

The next figure gives a compact architecture-by-architecture view of three headline quantities:
final loss, F/W ratio, and conditioning. This is a descriptive comparison, not a significance test.

In [ ]:
h3.plot_combined_summary(results, show=True)

In [ ]:
linear_muon = results['architectures']['linear']['muon']
linear_norm = results['architectures']['linear']['norm_sgd']
relu_muon = results['architectures']['relu']['muon']
relu_norm = results['architectures']['relu']['norm_sgd']
relu_best_ratio_method = results['heuristic_results']['relu']['best_ratio_method']
linear_best_ratio_method = results['heuristic_results']['linear']['best_ratio_method']

conclusion_md = f"""
## Calibrated conclusion

### Linear architecture
- Muon final mean loss: **{linear_muon['loss_mean']:.6e}**
- Normalized SGD final mean loss: **{linear_norm['loss_mean']:.6e}**
- Lowest F/W ratio among normalization-style methods: **{h3.OPTIMIZER_LABELS[linear_best_ratio_method]}**
- Interpretation: {results['heuristic_results']['linear']['critical_comparison']['conclusion']}

### ReLU architecture
- Muon final mean loss: **{relu_muon['loss_mean']:.6e}**
- Normalized SGD final mean loss: **{relu_norm['loss_mean']:.6e}**
- Lowest F/W ratio among normalization-style methods: **{h3.OPTIMIZER_LABELS[relu_best_ratio_method]}**
- Interpretation: {results['heuristic_results']['relu']['critical_comparison']['conclusion']}

### Overall reading
{results['overall_summary']['interpretation']}

### What this notebook does **not** establish
- no formal significance testing
- no direct measurement of gauge coordinates or orbit distance
- no direct directional-quality metric beyond the implemented loss and diversity summaries
- no universal statement about whether the polar factor is necessary outside this toy setup
"""

display(Markdown(conclusion_md))

## Practical next verification steps

For a stronger second pass, the most useful follow-up would be to add seed-level uncertainty intervals or bootstrap
summaries for selected metrics, and then re-check that the architecture-specific conclusions remain stable. But this
first completion pass already fixes the important execution/reporting problems:

- the notebook is notebook-safe and does not depend on `__file__`
- the experiment core is imported from the script rather than duplicated
- saved output paths are explicit
- figures are displayed inline instead of only being saved silently
- the narrative is calibrated to descriptive, architecture-dependent evidence